In [2]:
dataset = []
with open('gatos.txt', 'r', encoding='utf-8') as file:
  dataset = file.readlines()
  print(f'Loaded {len(dataset)} entries')


Loaded 196 entries


In [1]:
# %pip install ollama
import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
  
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))


In [3]:
for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')


Added chunk 1/196 to the database
Added chunk 2/196 to the database
Added chunk 3/196 to the database
Added chunk 4/196 to the database
Added chunk 5/196 to the database
Added chunk 6/196 to the database
Added chunk 7/196 to the database
Added chunk 8/196 to the database
Added chunk 9/196 to the database
Added chunk 10/196 to the database
Added chunk 11/196 to the database
Added chunk 12/196 to the database
Added chunk 13/196 to the database
Added chunk 14/196 to the database
Added chunk 15/196 to the database
Added chunk 16/196 to the database
Added chunk 17/196 to the database
Added chunk 18/196 to the database
Added chunk 19/196 to the database
Added chunk 20/196 to the database
Added chunk 21/196 to the database
Added chunk 22/196 to the database
Added chunk 23/196 to the database
Added chunk 24/196 to the database
Added chunk 25/196 to the database
Added chunk 26/196 to the database
Added chunk 27/196 to the database
Added chunk 28/196 to the database
Added chunk 29/196 to the dat

In [6]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)


In [4]:
def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order, because higher similarity means more relevant chunks
  similarities.sort(key=lambda x: x[1], reverse=True)
  # finally, return the top N most relevant chunks
  return similarities[:top_n]


In [10]:
input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

instruction_prompt = f'''You are veterenario y work in Argentina.:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''

Retrieved knowledge:
 - (similarity: 0.55) xet

 - (similarity: 0.54) raw

 - (similarity: 0.53) 22.7 kB



In [11]:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)


Chatbot response:
Excelente pregunta!

En general, las tabernas en Argentina tienen una vida promedio de aproximadamente 12 a 15 años.

Sin embargo, es importante destacar que la vida útil de un gato puede variar dependiendo de varios factores, como:

* La dieta: Un gato bien alimentado tendrá una vida más larga que uno mal alimentado.
* El estilo de vida: Los gatos felices y saludables vivirán más años que aquellos que sufren problemas de salud o viven en condiciones de abandono.
* La genes: Algunas razas de gatos, como la Persa, pueden tener una vida más larga que otras.

En resumen, aunque no hay un límite exacto para cuánto año viven los gatos, la mayoría de las tabernas en Argentina tienen un promedio de 12 a 15 años.

In [ ]:
''